In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler, Dataset
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from tqdm import tqdm
from collections import Counter
import random

# ==========================================
# REPRODUCIBILITY SEED
# ==========================================
def set_seed(seed=10):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ==========================================
# 0. Dataset Wrapper for Safe Transforms
# ==========================================
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# ==========================================
# 1. Gated Fusion Block
# ==========================================
class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.Sigmoid()
        )
        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x_deep, x_shallow):
        g = self.gate(x_deep)
        fused = x_deep + (g * self.proj(x_shallow))
        return self.norm(fused)

class SwinV2GatedModel(nn.Module):
    def __init__(self, num_classes=3, embed_dim=768, model_name='swinv2_tiny_window16_256.ms_in1k'):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, features_only=True, drop_path_rate=0.3)
        
        with torch.no_grad():
            dummy_out = self.backbone(torch.zeros(1, 3, 256, 256))
            is_nhwc = dummy_out[-1].shape[-1] > dummy_out[-1].shape[1]
            s3_channels = dummy_out[-2].shape[-1] if is_nhwc else dummy_out[-2].shape[1]
            s4_channels = dummy_out[-1].shape[-1] if is_nhwc else dummy_out[-1].shape[1]

        self.is_nhwc = is_nhwc
        self.proj3 = nn.Linear(s3_channels, embed_dim)
        self.spatial_downsample = nn.Conv2d(s3_channels, s3_channels, kernel_size=2, stride=2)
        self.fusion = GatedFusion(embed_dim)
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.5),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x, return_features=False):
        feats = self.backbone(x)
        s3_raw, s4_raw = feats[-2], feats[-1]

        if self.is_nhwc:
            s3_raw = s3_raw.permute(0, 3, 1, 2)
            s4_raw = s4_raw.permute(0, 3, 1, 2)

        s3_aligned = self.spatial_downsample(s3_raw) 
        s3_tokens = s3_aligned.flatten(2).transpose(1, 2)
        s4_tokens = s4_raw.flatten(2).transpose(1, 2)

        fused = self.fusion(s4_tokens, self.proj3(s3_tokens))
        pooled = fused.mean(dim=1)
        
        logits = self.classifier(pooled)
        
        if return_features:
            backbone_pooled = s4_tokens.mean(dim=1)
            return logits, backbone_pooled
        return logits

# ==========================================
# 2. Artifact Generation & Visualizations
# ==========================================
class HookHelper:
    def __init__(self, module):
        self.hook_f = module.register_forward_hook(self.hook_fn_fwd)
        self.hook_b = module.register_full_backward_hook(self.hook_fn_bwd)
        self.acts, self.grads = None, None

    def hook_fn_fwd(self, module, input, output): self.acts = output
    def hook_fn_bwd(self, module, grad_input, grad_output): self.grads = grad_output[0]
    def remove(self): self.hook_f.remove(); self.hook_b.remove()

def save_tsne(features, labels, classes, filename):
    print(f"\n--- Generating t-SNE Plot: {filename} ---")
    tsne = TSNE(n_components=2, random_state=32, init='pca', learning_rate='auto')
    feats_2d = tsne.fit_transform(features)
    
    plt.figure(figsize=(9, 6))
    colors = sns.color_palette("hls", len(classes))
    for i, class_name in enumerate(classes):
        mask = labels == i
        plt.scatter(feats_2d[mask, 0], feats_2d[mask, 1], label=class_name, alpha=0.6, edgecolors='w', s=60, color=colors[i])
    
    plt.legend(loc='best', fontsize=14)
    plt.xlabel('Dimension 1', fontsize=14)
    plt.ylabel('Dimension 2', fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

def save_artifacts(model, loader, device, classes, history, n_classes):
    print("\n--- Final Evaluation & Visualizations ---")
    model.load_state_dict(torch.load('outputs/best_swin_v2_3class.pth', weights_only=True))
    model.eval()
    
    y_true, y_pred, y_prob, all_backbone, all_logits = [], [], [], [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Testing"):
            imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
            outs, backbone_feats = model(imgs, return_features=True)
            y_prob.extend(torch.softmax(outs, dim=1).cpu().numpy())
            y_pred.extend(outs.argmax(1).cpu().numpy())
            y_true.extend(lbls.cpu().numpy())
            all_backbone.extend(backbone_feats.cpu().numpy())
            all_logits.extend(outs.cpu().numpy())

    print("\nClassification Report:\n" + classification_report(y_true, y_pred, target_names=classes, digits=4))
    
    # Save Loss/Acc Curves
    for key, name in [('loss', 'Loss'), ('acc', 'Accuracy')]:
        plt.figure(figsize=(8, 6))
        plt.plot(history[f'train_{key}'], label=f'Train {name}', lw=2)
        plt.plot(history[f'val_{key}'], label=f'Val {name}', lw=2)
        plt.xlabel('Epochs'); plt.ylabel(name); plt.legend(); plt.grid(True, alpha=0.3)
        plt.savefig(f'outputs/{key}_curve.png', dpi=300); plt.close()

    save_tsne(np.array(all_backbone), np.array(y_true), classes, 'outputs/tsne_backbone.png')
    save_tsne(np.array(all_logits), np.array(y_true), classes, 'outputs/tsne_classifier.png')

# ==========================================
# 3. Training Pipeline
# ==========================================
def train_and_evaluate(data_dir, epochs=30, batch_size=16):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs('outputs', exist_ok=True)
    
    model = SwinV2GatedModel(num_classes=3).to(device)
    
    # Data Setup
    tr_trans = transforms.Compose([
        transforms.RandomResizedCrop(256, scale=(0.7, 1.0)),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    ev_trans = transforms.Compose([
        transforms.Resize(288), transforms.CenterCrop(256), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    ds = ImageFolder(data_dir)
    allowed = ['Healthy', 'Diabetic Retinopathy', 'Glaucoma']
    filtered_samples = [(p, {c: i for i, c in enumerate([c for c in ds.classes if c in allowed])}[ds.classes[t]]) 
                        for p, t in ds.samples if ds.classes[t] in allowed]
    ds.classes, ds.samples = [c for c in ds.classes if c in allowed], filtered_samples
    ds.targets = [s[1] for s in filtered_samples]

    n = len(ds)
    tr_ds_raw, v_ds_raw, te_ds_raw = random_split(ds, [int(0.7*n), int(0.15*n), n - int(0.7*n) - int(0.15*n)])
    tr_targets = [ds.targets[i] for i in tr_ds_raw.indices]
    sampler = WeightedRandomSampler(1./torch.bincount(torch.tensor(tr_targets)).float()[tr_targets], len(tr_targets), replacement=True)

    train_loader = DataLoader(TransformSubset(tr_ds_raw, tr_trans), batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = DataLoader(TransformSubset(v_ds_raw, ev_trans), batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    te_loader = DataLoader(TransformSubset(te_ds_raw, ev_trans), batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # Optimizer & Tools
    optimizer = optim.AdamW([{'params': model.backbone.parameters(), 'lr': 1e-5}, 
                             {'params': [p for n, p in model.named_parameters() if 'backbone' not in n], 'lr': 5e-4}], weight_decay=0.05)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.15).to(device)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')
    
    best_v_loss, history = float('inf'), {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    print(f"\nStarting Training on {device}...")
    print(f"{'Epoch':<8} | {'Tr-Loss':<8} | {'Tr-Acc':<8} | {'Val-Loss':<8} | {'Val-Acc':<8}")
    print("-" * 55)

    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        r_loss, r_corr, r_tot = 0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.autocast('cuda', dtype=torch.float16):
                outs = model(imgs); loss = criterion(outs, lbls)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer) # Required for clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Stability fix
            scaler.step(optimizer)
            scaler.update()
            
            r_loss += loss.item() * imgs.size(0); r_tot += lbls.size(0); r_corr += (outs.argmax(1) == lbls).sum().item()

        t_loss, t_acc = r_loss/r_tot, r_corr/r_tot

        # --- VALIDATION PHASE ---
        model.eval()
        v_loss_sum, v_corr, v_tot = 0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
                with torch.autocast('cuda', dtype=torch.float16):
                    outs = model(imgs); loss = criterion(outs, lbls)
                v_loss_sum += loss.item() * imgs.size(0); v_tot += lbls.size(0); v_corr += (outs.argmax(1) == lbls).sum().item()
        
        v_loss, v_acc = v_loss_sum/v_tot, v_corr/v_tot
        
        # Log History
        history['train_loss'].append(t_loss); history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc); history['val_acc'].append(v_acc)
        scheduler.step()

        # Display Metrics
        print(f"{epoch+1:02d}/{epochs:02d}   | {t_loss:.4f}  | {t_acc:.4f}  | {v_loss:.4f}  | {v_acc:.4f}", end="")
        
       
        if v_loss < best_v_loss:
            best_v_loss = v_loss
            torch.save(model.state_dict(), 'outputs/best_swin_v2_3class.pth')
            print("  <-- Validation Improved. Saved!")
        else:
            print("") # New line if no improvement

    save_artifacts(model, te_loader, device, ds.classes, history, len(ds.classes))

if __name__ == '__main__':
    set_seed(10)
    train_and_evaluate('/kaggle/input/datasets/akv100685/retinal-5-class/Retinal_Dataset')